In [161]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
df_train = pd.read_csv('cars_ml_8k.csv')
df_train.head()




,brand,model,year,price,city,mileage_km,body_type,engine_volume_l,fuel_type,transmission,drive_type,steering_wheel,color,condition,generation
0,Toyota,Highlander Luxe,2017,17590000,Астана,230186.0,Кроссовер,3.5,бензин,Автомат,Полный привод,Слева,черный,NaN,2016 - 2019 3 поколение рестайлинг (U5)
1,Rox,01 VIP,2026,30490000,Алматы,NaN,Внедорожник,1.5,гибрид,Автомат,Полный привод,Слева,NaN,NaN,2023 - н.в. 1 поколение
2,Hyundai,Elantra,2020,7700000,Алматы,183426.0,Седан,2.0,бензин,Автомат,Передний привод,Слева,серый металлик,NaN,2018 - 2020 6 поколение рестайлинг (AD/ADA)
3,Kia,Sportage,2014,7000000,Петропавловск,114500.0,Кроссовер,2.0,бензин,Механика,Передний привод,Слева,серый,NaN,2010 - 2014 3 поколение (SL)
4,Toyota,Land Cruiser Prado,2014,14999999,Актобе,NaN,Внедорожник,2.7,бензин,Автомат,Полный привод,Слева,NaN,NaN,2013 - 2017 J150 рестайлинг


In [163]:
CURRENT_YEAR = 2026
import re
def split_generation(text):
    text = str(text).lower()

    years = re.findall(r"\d{4}", text)

    start_year = int(years[0]) if len(years) >= 1 else np.nan

    if "н.в" in text:
        end_year = CURRENT_YEAR
    else:
        end_year = int(years[1]) if len(years) >= 2 else np.nan

    gen = re.search(r"(\d+)\s*покол", text)
    generation_number = int(gen.group(1)) if gen else np.nan

    is_current = 1 if "н.в" in text else 0

    return pd.Series([
        start_year,
        end_year,
        generation_number,
        is_current
    ])


def process_data(df):
    df = df.copy()
    df['price'] = np.log1p(df['price'])
    df[[
        "generation_start_year",
        "generation_number"
  
    ]] = df["generation"].apply(split_generation)

    #drop
    df = df.drop("generation", axis=1)
    df = df.drop("condition", axis=1)
    df = df.drop("color", axis=1)
    df = df.drop("city", axis=1)
    df = df.dropna(subset=["brand", "model", "fuel_type"])


    #fillna
    df['engine_volume_l'] = df['engine_volume_l'].fillna(df['engine_volume_l'].median())
    #df['has_gen_number'] = df['generation_number'].apply(lambda x: 0 if pd.isna(x) else 1)
    df['mileage_km'] = df['mileage_km'].fillna(0)
    df['is_new'] = df['mileage_km'].apply(lambda x: 1 if x == 0 else 0)
    df['generation_start_year'] = df['generation_start_year'].fillna(0)
    
    

    return df

df_train = process_data(df_train)
df_train.tail()



ValueError: Columns must be same length as key

In [155]:
df_train.head()

,brand,model,year,price,mileage_km,body_type,engine_volume_l,fuel_type,transmission,drive_type,steering_wheel,generation_start_year,is_new
0,Toyota,Highlander Luxe,2017,16.682841,230186.0,Кроссовер,3.5,бензин,Автомат,Полный привод,Слева,2016.0,0
1,Rox,01 VIP,2026,17.232909,0.0,Внедорожник,1.5,гибрид,Автомат,Полный привод,Слева,2023.0,1
2,Hyundai,Elantra,2020,15.856731,183426.0,Седан,2.0,бензин,Автомат,Передний привод,Слева,2018.0,0
3,Kia,Sportage,2014,15.761421,114500.0,Кроссовер,2.0,бензин,Механика,Передний привод,Слева,2010.0,0
4,Toyota,Land Cruiser Prado,2014,16.523561,0.0,Внедорожник,2.7,бензин,Автомат,Полный привод,Слева,2013.0,1


In [154]:
import numpy as np
from catboost import CatBoostRegressor
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X,y = df_train.drop('price', axis=1), df_train['price']
cat_columns = [
    "model",
    "city",
    'brand',
    "body_type",
    "fuel_type",
    "transmission",
    "drive_type",
    "steering_wheel",
    
]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = CatBoostRegressor(
    iterations=10000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    loss_function="RMSE",
    random_seed=42,
    early_stopping_rounds=300,
    verbose=500
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_columns,
    eval_set=(X_test, y_test),
    use_best_model=True
)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f'MAE: {mae:.4f}')
print(f'MSE: {mse:.4f}')
print(f'R²: {r2:.4f}')  

ValueError: list.index(x): x not in list

In [ ]:
manual_car = pd.DataFrame([{
    "model": "corolla",
    'brand': "Toyota",
    "year": 2017,
    "body_type": "седан",
    "engine_volume_l": 1.6,
    "fuel_type": "бензин",
    "transmission": "Автомат",
    "drive_type": "передний привод",
    "steering_wheel": "Справа",
    "mileage_km": "200",
    "is_new": 0,
    "generation_start_year": 2019,
    
   
}])

manual_car = manual_car.reindex(columns=X_train.columns, fill_value=0)
for col in cat_columns:
    manual_car[col] = manual_car[col].astype(str).fillna("unknown")
pred_log = model.predict(manual_car)

pred_price = np.expm1(pred_log)

print(f"Predicted price: {pred_price[0]:,.0f} ₸")

Predicted price: 9,846,716 ₸


In [123]:
df_train['brand'].value_counts()

brand
Toyota           882
Hyundai          504
ВАЗ              362
Kia              316
Mercedes-Benz    293
                ... 
BAW                1
Foton              1
Saab               1
Maserati           1
Iran               1
Name: count, Length: 93, dtype: int64